In [5]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor

# 1. Load Dataset Utama
df = pd.read_excel("C:/Users/Lenovo/OneDrive/ML/data/ml_electrocoagulation.xlsx", sheet_name="Analysis_dataset")

X_base_num = ["voltage", "treatment_time", "current_density", "electrode_gap"]
X_base_cat = ["electrode_type", "electrolyte_type"]

y_turb = df["turbidity_final"]
y_filt = df["filtration_final"]

# 2. Formulasi Fitur Input
X_turb = df[X_base_num]
X_filt = pd.get_dummies(df[X_base_num + X_base_cat], columns=X_base_cat, dtype=int)

# 3. Fit Pipeline Final (Scaler + Model Terbaik)
pipe_turb = Pipeline([
    ("scaler", StandardScaler()), 
    ("model", GradientBoostingRegressor(n_estimators=100, random_state=42))
])
pipe_turb.fit(X_turb, y_turb)

pipe_filt = Pipeline([
    ("scaler", StandardScaler()), 
    ("model", RandomForestRegressor(n_estimators=100, random_state=42))
])
pipe_filt.fit(X_filt, y_filt)

# 4. Simpan ke Artifact Joblib
artifacts = {
    "turbidity_model": pipe_turb,
    "filtration_model": pipe_filt,
    "feature_columns_turb": list(X_turb.columns),
    "feature_columns_filt": list(X_filt.columns),
    "metrics_loso": {
        "turbidity": {"algorithm": "Gradient Boosting", "R2": 0.6020, "MAE": 6.02, "RMSE": 8.62, "NRMSE_pct": 12.20},
        "filtration": {"algorithm": "Random Forest", "R2": 0.6211, "MAE": 0.0335, "RMSE": 0.0425, "NRMSE_pct": 14.65}
    }
}

joblib.dump(artifacts, "m3tb_dss_models.joblib")
print("✅ File 'm3tb_dss_models.joblib' berhasil diperbarui dan siap digunakan di Streamlit!")

✅ File 'm3tb_dss_models.joblib' berhasil diperbarui dan siap digunakan di Streamlit!


In [6]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

print("Library berhasil dimuat!")
# Memuat data dari file Excel
file_path = "C:/Users/Lenovo/OneDrive/ML/data/ml_electrocoagulation.xlsx"
df = pd.read_excel(file_path, sheet_name="Analysis_dataset")

# Grouping berdasarkan studi/peneliti untuk LOSO-CV
groups = df["study_id"]

# Target Variabel
y_turb = df["turbidity_final"]
y_filt = df["filtration_final"]

# Definisi Fitur Input
X_base_num = ["voltage", "treatment_time", "current_density", "electrode_gap"]
X_base_cat = ["electrode_type", "electrolyte_type"]

# Formulasi Matriks Fitur X
X_turb = df[X_base_num]
X_filt = pd.get_dummies(df[X_base_num + X_base_cat], columns=X_base_cat, dtype=int)

print(f"Dataset berhasil dimuat: Total {len(df)} observasi data.")
logo = LeaveOneGroupOut()

oof_turb = np.zeros(len(df))
oof_filt = np.zeros(len(df))

# 1. Validasi LOSO-CV Model Turbiditas (Gradient Boosting)
for train_idx, test_idx in logo.split(X_turb, y_turb, groups):
    pipe_tb = Pipeline([
        ("scaler", StandardScaler()),
        ("model", GradientBoostingRegressor(n_estimators=100, random_state=42))
    ])
    pipe_tb.fit(X_turb.iloc[train_idx], y_turb.iloc[train_idx])
    oof_turb[test_idx] = pipe_tb.predict(X_turb.iloc[test_idx])

# 2. Validasi LOSO-CV Model Kecepatan Filtrasi (Random Forest)
for train_idx, test_idx in logo.split(X_filt, y_filt, groups):
    pipe_fl = Pipeline([
        ("scaler", StandardScaler()),
        ("model", RandomForestRegressor(n_estimators=100, random_state=42))
    ])
    pipe_fl.fit(X_filt.iloc[train_idx], y_filt.iloc[train_idx])
    oof_filt[test_idx] = pipe_fl.predict(X_filt.iloc[test_idx])

# Kalkulasi Metrik Evaluasi
r2_turb = r2_score(y_turb, oof_turb)
mae_turb = mean_absolute_error(y_turb, oof_turb)
rmse_turb = np.sqrt(mean_squared_error(y_turb, oof_turb))
nrmse_turb = (rmse_turb / (y_turb.max() - y_turb.min())) * 100

r2_filt = r2_score(y_filt, oof_filt)
mae_filt = mean_absolute_error(y_filt, oof_filt)
rmse_filt = np.sqrt(mean_squared_error(y_filt, oof_filt))
nrmse_filt = (rmse_filt / (y_filt.max() - y_filt.min())) * 100

print("=== METRIK EVALUASI MODEL (LOSO-CV BENCHMARK) ===")
print(f"Turbiditas (GBR) -> R2: {r2_turb:.4f} | MAE: {mae_turb:.2f} NTU | RMSE: {rmse_turb:.2f} NTU | NRMSE: {nrmse_turb:.2f}%")
print(f"Filtrasi (RF)   -> R2: {r2_filt:.4f} | MAE: {mae_filt:.4f} mL/s | RMSE: {rmse_filt:.4f} mL/s | NRMSE: {nrmse_filt:.2f}%")
# Pelatihan model pada seluruh 58 data untuk deployment Streamlit
pipe_turb_full = Pipeline([
    ("scaler", StandardScaler()),
    ("model", GradientBoostingRegressor(n_estimators=100, random_state=42))
])
pipe_turb_full.fit(X_turb, y_turb)

pipe_filt_full = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(n_estimators=100, random_state=42))
])
pipe_filt_full.fit(X_filt, y_filt)

# Simpan artifact model
artifacts = {
    "turbidity_model": pipe_turb_full,
    "filtration_model": pipe_filt_full,
    "feature_columns_turb": list(X_turb.columns),
    "feature_columns_filt": list(X_filt.columns)
}

joblib_path = os.path.join(os.getcwd(), "m3tb_dss_models.joblib")
joblib.dump(artifacts, joblib_path)

print(f"✅ Model berhasil di-export ke: {joblib_path}")
# Prediksi menggunakan full model
pred_turb_full = pipe_turb_full.predict(X_turb)
pred_filt_full = pipe_filt_full.predict(X_filt)

# Menyusun dataframe komparasi 58 data
df_komparasi = pd.DataFrame({
    "No": range(1, len(df) + 1),
    "Study_ID": df["study_id"],
    "Voltage_V": df["voltage"],
    "Time_min": df["treatment_time"],
    "Gap_cm": df["electrode_gap"],
    "Electrode": df["electrode_type"],
    "Electrolyte": df["electrolyte_type"],
    "Turbiditas_Aktual": y_turb,
    "Turbiditas_Prediksi": np.round(pred_turb_full, 2),
    "Selisih_Turbiditas": np.round(pred_turb_full - y_turb, 2),
    "Filtrasi_Aktual": y_filt,
    "Filtrasi_Prediksi": np.round(pred_filt_full, 4),
    "Selisih_Filtrasi": np.round(pred_filt_full - y_filt, 4)
})

# Ekspor tabel ke file CSV
csv_path = "tabel_perbandingan_58_data.csv"
df_komparasi.to_csv(csv_path, index=False)

print(f"✅ Tabel perbandingan 58 data berhasil disimpan ke: {csv_path}")

# Tampilkan seluruh 58 data di notebook
pd.set_option('display.max_rows', 60)
df_komparasi


Library berhasil dimuat!
Dataset berhasil dimuat: Total 58 observasi data.
=== METRIK EVALUASI MODEL (LOSO-CV BENCHMARK) ===
Turbiditas (GBR) -> R2: 0.6020 | MAE: 6.02 NTU | RMSE: 8.62 NTU | NRMSE: 12.20%
Filtrasi (RF)   -> R2: 0.6230 | MAE: 0.0334 mL/s | RMSE: 0.0424 mL/s | NRMSE: 14.61%
✅ Model berhasil di-export ke: c:\Users\Lenovo\OneDrive\ML\notebooks\m3tb_dss_models.joblib
✅ Tabel perbandingan 58 data berhasil disimpan ke: tabel_perbandingan_58_data.csv


,No,Study_ID,Voltage_V,Time_min,Gap_cm,Electrode,Electrolyte,Turbiditas_Aktual,Turbiditas_Prediksi,Selisih_Turbiditas,Filtrasi_Aktual,Filtrasi_Prediksi,Selisih_Filtrasi
0,1,Cipek1,30,5,1.5,Al - Al,NaCl,23.64,24.03,0.39,0.17,0.1653,-0.0047
1,2,Cipek2,30,5,1.5,Al - Al,NaCl,10.63,10.50,-0.13,0.16,0.1636,0.0036
2,3,Cipek3,30,5,1.5,Al - Al,NaCl,7.51,10.61,3.10,0.15,0.1649,0.0149
3,4,Cipek4,30,5,1.5,Al - Al,NaCl,3.40,6.82,3.42,0.14,0.1643,0.0243
4,5,Cipek5,30,5,1.5,Al - Al,NaCl,3.64,5.37,1.73,0.19,0.1867,-0.0033
5,6,Cipek6,30,5,1.5,Al - Al,NaCl,5.39,6.51,1.12,0.25,0.2346,-0.0154
6,7,Cipek7,30,5,1.5,Al - Al,NaCl,5.51,6.28,0.77,0.24,0.2360,-0.0040
7,8,Cipek8,30,5,1.5,Al - Al,Na2SO4,9.55,11.34,1.79,0.30,0.2773,-0.0227
8,9,Cipek9,30,5,1.5,Al - Al,Na2SO4,4.85,6.89,2.04,0.31,0.2838,-0.0262
9,10,Cipek10,30,5,1.5,Al - Al,Na2SO4,5.80,9.71,3.91,0.32,0.2923,-0.0277
